# 2.8 — The Map-Side (Broadcast) Join

**Chapter 2, section 2.9.1** (*The Map-Side (Broadcast) Join*).

**The question this notebook answers:** a join is a shuffle, and a shuffle is the most
expensive thing in a job — so how do you get the join without the shuffle?

The chapter calls this "the highest-value optimization in this chapter", and the mechanism is
simple: if one side is small enough to fit in an executor's memory, send a copy of it to every
executor and join with a plain `map`. No record of the large side moves at all.

Three strategies are measured below — the ordinary shuffled join, the broadcast join, and the
intermediate one for when the small side is *nearly* small enough — with the shuffle bytes read
from Spark's own metrics, so that "no shuffle" is a number rather than a claim.

Section 4 is about `sc.broadcast` itself, and why the chapter insists on it rather than on
referring to the Python variable directly.

Covers **Exercise 6**.

Runs on a laptop in about a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
# Chapter 2, section 2.9.1.
import os, json, time, random, tempfile, urllib.request
from pyspark.sql import SparkSession
from pyspark.serializers import CloudPickleSerializer

SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-2.8")
         .master("local[*]")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext
UI = sc.uiWebUrl
APP = json.load(urllib.request.urlopen(UI + "/api/v1/applications"))[0]["id"]

def _get(path):
    return json.load(urllib.request.urlopen(f"{UI}/api/v1/applications/{APP}{path}"))

def run(label, fn):
    """Run an action under a job group; report time and total shuffle bytes written."""
    sc.setJobGroup(label, label)
    t = time.perf_counter()
    out = fn()
    ms = (time.perf_counter() - t) * 1000
    stage_ids = sorted({s for j in _get("/jobs") if j.get("jobGroup") == label
                        for s in j["stageIds"]})
    shuffled = sum(_get(f"/stages/{sid}/0")["shuffleWriteBytes"] for sid in stage_ids)
    return {"label": label, "ms": ms, "result": out, "shuffle": shuffled}

print("Spark", spark.version, "on", sc.master)

Spark 4.2.0 on local[*]


## 1. The two sides

The shape of the problem, in miniature: a large table keyed by store, and a small lookup table
of store metadata. This is Exercise 6's 500 GB ⋈ 2 MB, scaled down until it runs on a laptop.

In [2]:
STORES = 200
N = 2_000_000

def make_transactions(idx, it):
    rnd = random.Random(4000 + idx)
    for _ in it:
        yield (f"store-{rnd.randrange(STORES):04d}", round(rnd.random() * 100, 2))

large = sc.range(0, N, numSlices=8).mapPartitionsWithIndex(make_transactions).cache()
print(f"large : {large.count():,} transactions over {STORES} stores")

rnd = random.Random(11)
small = sc.parallelize(
    [(f"store-{i:04d}", f"City{rnd.randrange(50)}") for i in range(STORES)], 2)
print(f"small : {small.count()} store records")
print("sample:", small.take(3))

large : 2,000,000 transactions over 200 stores
small : 200 store records
sample: [('store-0000', 'City28'), ('store-0001', 'City35'), ('store-0002', 'City49')]


## 2. The ordinary join: both sides shuffle

`join` is a wide dependency. Both sides must be partitioned by the key so that matching
records meet, which means every record of the large side crosses the network.

In [3]:
shuffled = run("shuffled-join", lambda: large.join(small).count())
print(f"rows joined     : {shuffled['result']:,}")
print(f"time            : {shuffled['ms']:,.0f} ms")
print(f"shuffle written : {shuffled['shuffle']:,} bytes")

rows joined     : 2,000,000
time            : 481 ms
shuffle written : 9,664,003 bytes


## 3. The map-side join: nothing moves

Collect the small side into an ordinary Python dict, broadcast it once to every executor, and
the join becomes a `map` over the large side.

The two lines to read carefully are `bc = sc.broadcast(small_map)` and `bc.value` inside the
lambda. **Binding the result and using `bc.value` is the point**; calling `sc.broadcast(...)`
and then referring to the plain Python variable instead looks identical and does something
different, which section 4 is about.

In [4]:
small_map = small.collectAsMap()          # small enough for the driver, by construction
bc = sc.broadcast(small_map)              # ...and transferred to each executor ONCE

def map_side_join(kv):
    key, value = kv
    return (key, (value, bc.value[key]))

mapside = run("map-side-join", lambda: (large
    .filter(lambda kv: kv[0] in bc.value)
    .map(map_side_join)
    .count()))

print(f"rows joined     : {mapside['result']:,}")
print(f"time            : {mapside['ms']:,.0f} ms")
print(f"shuffle written : {mapside['shuffle']:,} bytes")

rows joined     : 2,000,000
time            : 110 ms
shuffle written : 0 bytes


In [5]:
# The two must agree, or the optimization is not an optimization.
assert shuffled["result"] == mapside["result"]
print(f"both strategies joined {mapside['result']:,} rows\n")

print(f"{'strategy':22s}{'ms':>10s}{'shuffle bytes':>16s}")
print(f"{'shuffled join':22s}{shuffled['ms']:>10,.0f}{shuffled['shuffle']:>16,}")
print(f"{'map-side (broadcast)':22s}{mapside['ms']:>10,.0f}{mapside['shuffle']:>16,}")
print(f"\n{shuffled['ms'] / mapside['ms']:.1f}x faster, and the shuffle is GONE -- not")
print("reduced, not compressed: zero bytes, because there is no wide dependency left.")
print("The join became a narrow transformation, and a narrow transformation runs inside")
print("its partition.")

both strategies joined 2,000,000 rows

strategy                      ms   shuffle bytes
shuffled join                481       9,664,003
map-side (broadcast)         110               0

4.4x faster, and the shuffle is GONE -- not
reduced, not compressed: zero bytes, because there is no wide dependency left.
The join became a narrow transformation, and a narrow transformation runs inside
its partition.


## 4. Why `sc.broadcast`, and not just the variable

Both of these appear to work:

```python
bc = sc.broadcast(small_map)
rdd.map(lambda kv: (kv[0], bc.value[kv[0]]))     # a broadcast variable

rdd.map(lambda kv: (kv[0], small_map[kv[0]]))    # the plain Python variable
```

The chapter's argument is that the second captures `small_map` in the function's **closure** —
the set of outside variables a function refers to, which Spark serializes and ships with the
function itself — so it is carried inside the task description of every stage that refers to
it, and **each task deserializes its own private copy**. A broadcast variable is transferred to
an executor once, deserialized once, and shared by every task that executor runs.

That is correct, and there is a PySpark-specific detail worth adding, because it changes when
the advice bites hardest.

In [6]:
ser = CloudPickleSerializer()

def closure_bytes(table):
    """How large is the pickled closure of a lambda that captures `table`?"""
    fn = lambda kv: (kv[0], table[kv[0]])
    return len(ser.dumps(fn))

# The values must be DISTINCT objects.  A dict whose values are all the same string --
# `{k: "x" * 200 for ...}`, where the expression is constant-folded -- pickles to almost
# nothing, because pickle memoizes repeated references.  That is a nice illustration of
# why measuring the serialized size beats estimating it from the row count.
tiny = {f"store-{i:04d}": f"City{i}" for i in range(200)}
big_table = {f"store-{i:06d}": f"City{i}-" + "y" * (200 + i % 7) for i in range(20_000)}

THRESHOLD = 1 << 20        # PySpark's default: see pyspark.rdd._prepare_for_python_RDD
print(f"{'lookup table':16s}{'closure bytes':>16s}   auto-broadcast by PySpark?")
for name, tbl in (("200 stores", tiny), ("20,000 stores", big_table)):
    n = closure_bytes(tbl)
    auto = "YES -- it does it for you" if n > THRESHOLD else "no -- one copy per task"
    print(f"{name:16s}{n:>16,}   {auto}")

lookup table       closure bytes   auto-broadcast by PySpark?
200 stores                 5,160   no -- one copy per task
20,000 stores          4,610,227   YES -- it does it for you


**PySpark auto-broadcasts a closure larger than about 1 MB.** `_prepare_for_python_RDD`
pickles the command and, if the result exceeds the threshold, wraps it in a broadcast without
being asked. So the "one copy per task" cost the chapter describes is real *below* that
threshold and is handled automatically above it.

That does not make `sc.broadcast` optional. Three reasons remain, and they are the ones worth
carrying:

1. **Below the threshold there is no help at all.** A 200 KB lookup table used by a stage of
   500 tasks travels 500 times. That is the common case, and it is invisible.
2. **The automatic broadcast has the lifetime of the RDD that triggered it**, so the same table
   used in three different transformations is broadcast three times. An explicit
   `sc.broadcast` is created once and shared by all of them, across stages and across RDDs.
3. **It is explicit.** Relying on a size threshold means the behaviour of a program changes
   when the lookup table grows past a number nobody looked up.

In [7]:
# Reason 2, made concrete: one explicit broadcast serving three separate transformations.
b = sc.broadcast(small_map)

r1 = large.filter(lambda kv: kv[0] in b.value).count()
r2 = large.map(lambda kv: (b.value[kv[0]], kv[1])).keys().distinct().count()
r3 = large.filter(lambda kv: b.value[kv[0]].endswith("7")).count()

print(f"three transformations, one broadcast : {r1:,} / {r2} / {r3:,}")
print(f"bytes transferred to each executor   : once, {len(ser.dumps(small_map)):,}")
print("\nThe same three written against the plain Python dict would each carry their own")
print("copy inside their own task descriptions -- or, above the threshold, trigger their")
print("own separate automatic broadcast.")

b.unpersist()

three transformations, one broadcast : 2,000,000 / 48 / 130,186
bytes transferred to each executor   : once, 4,368

The same three written against the plain Python dict would each carry their own
copy inside their own task descriptions -- or, above the threshold, trigger their
own separate automatic broadcast.


## 5. When the small side is not quite small enough

If the small data set is too large to broadcast whole, its **keys** may still fit. Broadcast
the key set, filter the large side down to the rows that could possibly match, and then run an
ordinary shuffled join on the much smaller result.

This does not eliminate the shuffle. It reduces the volume the shuffle has to move, which is
usually the point.

In [8]:
# Only a quarter of the stores are in the lookup table this time, so the filter has
# something to remove -- which is the situation where this strategy pays.
partial = small.filter(lambda kv: int(kv[0][-4:]) % 4 == 0)
print(f"lookup table now covers {partial.count()} of {STORES} stores")

keys = set(partial.keys().collect())        # just the KEYS
bk = sc.broadcast(keys)

naive = run("naive-join", lambda: large.join(partial).count())
prefiltered = run("prefiltered-join", lambda: (large
    .filter(lambda kv: kv[0] in bk.value)   # broadcast filter: narrow, no shuffle
    .join(partial)                          # ...and now a much smaller shuffle
    .count()))

assert naive["result"] == prefiltered["result"]
print(f"\nboth joined {naive['result']:,} rows\n")
print(f"{'strategy':26s}{'ms':>10s}{'shuffle bytes':>16s}")
print(f"{'plain join':26s}{naive['ms']:>10,.0f}{naive['shuffle']:>16,}")
print(f"{'broadcast keys, then join':26s}{prefiltered['ms']:>10,.0f}{prefiltered['shuffle']:>16,}")
print(f"\nshuffle reduced {naive['shuffle'] / max(prefiltered['shuffle'], 1):.1f}x by "
      f"discarding non-matching rows BEFORE the shuffle rather than during it.")

bk.unpersist()

lookup table now covers 50 of 200 stores



both joined 500,304 rows

strategy                          ms   shuffle bytes
plain join                       329       9,662,538
broadcast keys, then join        166       2,433,836

shuffle reduced 4.0x by discarding non-matching rows BEFORE the shuffle rather than during it.


## 6. Exercise 6

> *You must join a 500 GB table of transactions, keyed by `store_id`, with a 2 MB table of
> store metadata. (a) Write the broadcast join in PySpark. (b) Estimate how much data crosses
> the network in your version and in a naive `join`. (c) At roughly what size of the small
> table would you stop broadcasting, and what would you do instead?*

In [9]:
print("(a)  small_map = small.collectAsMap()")
print("     bc = sc.broadcast(small_map)")
print("     joined = (large.filter(lambda kv: kv[0] in bc.value)")
print("                   .map(lambda kv: (kv[0], (kv[1], bc.value[kv[0]]))))")
print()

GB = 1024 ** 3
MB = 1024 ** 2
large_bytes, small_bytes, executors = 500 * GB, 2 * MB, 100

print("(b)  naive join     : both sides shuffle, so essentially the whole of both")
print(f"                      {large_bytes / GB:,.0f} GB + {small_bytes / MB:.0f} MB "
      f"= about {(large_bytes + small_bytes) / GB:,.0f} GB across the network")
print(f"     broadcast join : the small side only, once per executor")
print(f"                      {small_bytes / MB:.0f} MB x {executors} executors "
      f"= {small_bytes * executors / MB:,.0f} MB = "
      f"{small_bytes * executors / GB:.2f} GB")
print(f"                      ratio: about {(large_bytes) / (small_bytes * executors):,.0f}x "
      f"less data moved")
print()
print("(c)  Stop when a copy of the small table stops fitting comfortably in ONE executor's")
print("     memory alongside everything else that executor is doing -- in practice a few")
print("     hundred MB, and the DataFrame API's own default threshold is far lower still,")
print("     at 10 MB (spark.sql.autoBroadcastJoinThreshold).  Note the cost is per")
print("     executor, so a 500 MB table on 500 executors is 250 GB of broadcast traffic")
print("     and the arithmetic stops favouring it long before memory does.")
print()
print("     Beyond that size: broadcast the KEYS and pre-filter (section 5); or, if the")
print("     same join is run repeatedly, co-partition both sides once with partitionBy")
print("     on the join key and reuse the partitioning.")

(a)  small_map = small.collectAsMap()
     bc = sc.broadcast(small_map)
     joined = (large.filter(lambda kv: kv[0] in bc.value)
                   .map(lambda kv: (kv[0], (kv[1], bc.value[kv[0]]))))

(b)  naive join     : both sides shuffle, so essentially the whole of both
                      500 GB + 2 MB = about 500 GB across the network
     broadcast join : the small side only, once per executor
                      2 MB x 100 executors = 200 MB = 0.20 GB
                      ratio: about 2,560x less data moved

(c)  Stop when a copy of the small table stops fitting comfortably in ONE executor's
     memory alongside everything else that executor is doing -- in practice a few
     hundred MB, and the DataFrame API's own default threshold is far lower still,
     at 10 MB (spark.sql.autoBroadcastJoinThreshold).  Note the cost is per
     executor, so a 500 MB table on 500 executors is 250 GB of broadcast traffic
     and the arithmetic stops favouring it long before memory do

## Conclusion

**A join is a wide dependency, and a wide dependency is a shuffle.** Unless one side is small
enough to copy, in which case it is not a join at all any more — it is a `map` with a lookup
table, and it moves nothing.

The measurement in section 3 is the whole argument: the same answer, and the shuffle drops to
**zero bytes**. That is why the chapter calls it the highest-value optimization in the chapter,
and why the DataFrame API automates the same decision with
`spark.sql.autoBroadcastJoinThreshold` — which chapter 3 takes up, along with what to do when
the optimizer guesses the size wrong.

Three things to remember:

* **Bind the broadcast and use `.value`.** `sc.broadcast(x)` whose return value is thrown away
  does nothing; the lambda then captures the plain variable, and you have written the version
  the chapter is arguing against while appearing to have taken its advice.
* **The cost of broadcasting is per executor, not per job.** A table that is comfortable on ten
  executors may not be on a thousand.
* **If the whole table will not fit, its keys might.** Filtering the large side before the
  shuffle is not as good as removing the shuffle, but it is much better than nothing.

**Next.** Notebook 2.9 is the reference for the four join flavours themselves.